In [16]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from numpy.fft import fft2, ifft2, fftshift
import ipywidgets as widgets
from IPython.display import display
from scipy.ndimage import median_filter
from skimage.metrics import structural_similarity
from skimage.metrics import peak_signal_noise_ratio
import os,re,glob
import pandas as pd
from skimage.metrics import mean_squared_error
from skimage.feature import match_template
from PIL import Image


In [2]:
# 1. CARGAR CUBO FITS

def SpowerSpecleSpectrum(data,mostrarImagenes = False,MFBD = None):

    n_frames, H, W = data.shape

    # 1. FFT DE TODOS LOS FRAMES
    fft_frames = []
    for i in range(n_frames):
        frame = data[i]
        frame = frame.astype(np.float64)
        # opcional: quitar promedio
        frame = frame - np.mean(frame)
        F = fft2(frame)
        fft_frames.append(F)
    fft_frames = np.array(fft_frames)

    # 3. POWER SPECTRUM PROMEDIO

    power_spectrum = np.mean(np.abs(fft_frames) ** 2, axis=0)
    amplitude = np.sqrt(power_spectrum)

    # 4. RECONSTRUCCIÓN FINAL

    avg_fft = np.mean(fft_frames, axis=0)
    phase_avg = np.angle(avg_fft)
    reconstructed_fft = amplitude * np.exp(1j * phase_avg)
    reconstructed_image = np.real(ifft2(reconstructed_fft))

    # 5. VISUALIZACIÓN
    if mostrarImagenes == True:
        plt.figure(figsize=(15, 5))

        plt.subplot(2, 3, 1)
        plt.title("Frame original")
        plt.imshow(data[0])
        plt.colorbar()

        plt.subplot(2, 3, 2)
        plt.title("Power Spectrum")
        plt.imshow(np.log1p(fftshift(power_spectrum)))
        plt.colorbar()

        plt.subplot(2, 3, 3)
        plt.title("Reconstrucción")
        plt.imshow(reconstructed_image)
        plt.colorbar()

        mfbd = fits.getdata(MFBD)
        plt.subplot(2, 3, 4)
        plt.title("MFBD reconstruccion")
        plt.imshow(mfbd)
        plt.colorbar()

        plt.subplot(2, 3, 5)
        plt.title("Phase")
        plt.imshow(phase_avg)
        plt.colorbar()

        plt.tight_layout()
        plt.show()

    return reconstructed_image

In [ ]:
def procesar_y_normalizar(data):
    """
    Limpia valores NaN/Inf y normaliza la imagen al rango [0, 1].
    """
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
    
    min_val = data.min()
    max_val = data.max()
    
    if max_val == min_val:
        return np.zeros_like(data)
    
    return (data - min_val) / (max_val - min_val)


def alinear_y_recortar(img1, img2):
    """
    Encuentra la imagen más pequeña dentro de la más grande usando Correlación Cruzada.
    Retorna ambas imágenes recortadas para que abarquen exactamente la misma zona.
    """
    # 1. Determinar cuál actúa como 'molde' (template) y cuál como 'lienzo' (search space)
    if img1.size < img2.size:
        template, search_image = img1, img2
        es_img1_template = True
    else:
        template, search_image = img2, img1
        es_img1_template = False

    # Si ya son del mismo tamaño, no hay que alinear
    if template.shape == search_image.shape:
        return img1, img2

    # 2. Ejecutar la Correlación Cruzada Normalizada en 2D
    # Esto desliza el 'template' sobre el 'lienzo' y genera un mapa de calor de coincidencias
    mapa_correlacion = match_template(search_image, template)
    
    # 3. Encontrar las coordenadas (Y, X) del pico máximo de similitud
    y_start, x_start = np.unravel_index(np.argmax(mapa_correlacion), mapa_correlacion.shape)
    
    h, w = template.shape
    
    # 4. Recortar el lienzo para extraer exactamente la zona de coincidencia
    matched_crop = search_image[y_start : y_start + h, x_start : x_start + w]
    
    print(f"   * Coincidencia hallada: offset(Y={y_start}, X={x_start}). Alineación completada.")

    # 5. Devolver las matrices en su orden original para no alterar la referencia y el resultado
    if es_img1_template:
        return template, matched_crop
    else:
        return matched_crop, template
    
def digitos(numero,flag):
    if flag == True:
        numero += 100
    if numero > 99:
        return str(numero)
    elif numero > 9:
        return "0"+str(numero)
    else:
        return "00"+str(numero)


def comparar_directorios(Carpeta,TipoArchivo, csv_salida="resultados_metricas_alineados.csv", flag=False):
    resultados = []
    
    cantidad_fits = 100
    for i in range(cantidad_fits):
        number = digitos(i,flag)
        filename = Carpeta+"/"+TipoArchivo+number+".fits"
        mfbdname = Carpeta+"/"+TipoArchivo+number+"_MFBD.fits"

        images = fits.getdata(filename).astype(np.float32)
        mfbd   = fits.getdata(mfbdname).astype(np.float32)
        reconstructed = SpowerSpecleSpectrum(images)
    
        try:

            # --- EL CORAZÓN DE LA SOLUCIÓN: ALINEACIÓN POR CORRELACIÓN CRUZADA ---
            data_ref_alineada, data_res_alineada = alinear_y_recortar(mfbd, reconstructed)
            
            # Normalizar los recortes resultantes
            img_ref_norm = procesar_y_normalizar(data_ref_alineada)
            img_res_norm = procesar_y_normalizar(data_res_alineada)
            
            # Calcular métricas
            mse_val = mean_squared_error(img_ref_norm, img_res_norm)
            psnr_val = peak_signal_noise_ratio(img_ref_norm, img_res_norm, data_range=1.0)
            
            # Ajuste de seguridad dinámico para la ventana de SSIM
            win_size = min(7, img_ref_norm.shape[0], img_ref_norm.shape[1])
            win_size = win_size if win_size % 2 != 0 else win_size - 1
            
            ssim_val = structural_similarity(img_ref_norm, img_res_norm, data_range=1.0, win_size=win_size)
            
            resultados.append({
                "ID": TipoArchivo+number,
                "MSE": mse_val,
                "PSNR_dB": psnr_val,
                "SSIM": ssim_val
            })
            print(f"✅ ID {TipoArchivo+number} procesado | Tamaño validado: {img_ref_norm.shape} | SSIM: {ssim_val:.4f} | PSNR: {psnr_val:.2f} dB")
            
        except Exception as e:
            print(f"❌ Error crítico en ID {TipoArchivo+number}: {e}")

    # Exportar resultados tabulados
    if resultados:
        df = pd.DataFrame(resultados)
        df.to_csv(csv_salida, index=False)
        print(f"\n🎉 Proceso finalizado. Resultados exportados a: {os.path.abspath(csv_salida)}")
        
        print("\n--- Nuevo Resumen Promedio de Métricas ---")
        print(df[["MSE", "PSNR_dB", "SSIM"]].mean().to_string())
    else:
        print("\nNo se completó ninguna iteración válida.")


def GenerarResultados(CarpetaOrigen,TipoArchivo,NombreCarpetaDestino,flag=False):
    cantidad_fits = 100
    for i in range(cantidad_fits):
        number = digitos(i,flag)
        filename = CarpetaOrigen+"/"+TipoArchivo+number+".fits"
        images = fits.getdata(filename).astype(np.float32)
        reconstructed = SpowerSpecleSpectrum(images)
        img = reconstructed.copy()

        img = (img - img.min()) / (img.max() - img.min())
        img = (img * 255).astype(np.uint8)
        Image.fromarray(img).save(NombreCarpetaDestino+"/"+TipoArchivo+number+".png")



In [ ]:
Jose = "chromosphere_100p_0_99"
Benjamin = "chromosphere_100p_100_199"
Extra = "chromosphere_200p_0_99"
Nayely = "continuum_200p_0_99"
Joaquin = "continuum_200p_100_199"

ch100 = "chromosphere_100p_00" 
ch200 = "chromosphere_200p_00" 
co200 = "continuum_200p_00" 

In [ ]:
GenerarResultados(Jose,ch100,"Chromosfera100p-1",False)
GenerarResultados(Benjamin,ch100,"Chromosfera100p-2",True)
GenerarResultados(Nayely,co200,"Continuum100-1",False)
GenerarResultados(Joaquin,co200,"Continuum100p-2",True)
GenerarResultados(Extra,ch200,"Chromosfera200p",False)

In [ ]:

comparar_directorios(Joaquin,co200,"Dataset Joaquin.csv",True)

   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID continuum_200p_100_199/continuum_200p_00100.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.5563 | PSNR: 22.36 dB
   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID continuum_200p_100_199/continuum_200p_00101.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.8347 | PSNR: 34.98 dB
   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID continuum_200p_100_199/continuum_200p_00102.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.7695 | PSNR: 33.42 dB
   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID continuum_200p_100_199/continuum_200p_00103.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.8680 | PSNR: 35.74 dB
   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID continuum_200p_100_199/continuum_200p_00104.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.8309 | PSNR: 34.58 dB
   * Coincidencia ha

In [12]:
comparar_directorios(Nayely,co200,"Dataset Nayely.csv",False)

   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID continuum_200p_0_99/continuum_200p_00000.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.8347 | PSNR: 33.27 dB
   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID continuum_200p_0_99/continuum_200p_00001.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.7269 | PSNR: 24.23 dB
   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID continuum_200p_0_99/continuum_200p_00002.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.8074 | PSNR: 33.08 dB
   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID continuum_200p_0_99/continuum_200p_00003.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.7958 | PSNR: 31.70 dB
   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID continuum_200p_0_99/continuum_200p_00004.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.8351 | PSNR: 33.85 dB
   * Coincidencia hallada: offset(Y

In [13]:
comparar_directorios(Jose,ch100,"Dataset Jose.csv",False)

   * Coincidencia hallada: offset(Y=16, X=16). Alineación completada.
✅ ID chromosphere_100p_0_99/chromosphere_100p_00000.fits procesado | Tamaño validado: (70, 70) | SSIM: 0.6472 | PSNR: 22.04 dB
   * Coincidencia hallada: offset(Y=16, X=16). Alineación completada.
✅ ID chromosphere_100p_0_99/chromosphere_100p_00001.fits procesado | Tamaño validado: (70, 70) | SSIM: 0.6205 | PSNR: 21.17 dB
   * Coincidencia hallada: offset(Y=16, X=17). Alineación completada.
✅ ID chromosphere_100p_0_99/chromosphere_100p_00002.fits procesado | Tamaño validado: (70, 70) | SSIM: 0.5501 | PSNR: 18.17 dB
   * Coincidencia hallada: offset(Y=16, X=16). Alineación completada.
✅ ID chromosphere_100p_0_99/chromosphere_100p_00003.fits procesado | Tamaño validado: (70, 70) | SSIM: 0.5979 | PSNR: 19.87 dB
   * Coincidencia hallada: offset(Y=15, X=16). Alineación completada.
✅ ID chromosphere_100p_0_99/chromosphere_100p_00004.fits procesado | Tamaño validado: (70, 70) | SSIM: 0.5719 | PSNR: 24.10 dB
   * Coincidenc

In [14]:
comparar_directorios(Benjamin,ch100,"Dataset Benjamin.csv",True)

   * Coincidencia hallada: offset(Y=16, X=16). Alineación completada.
✅ ID chromosphere_100p_100_199/chromosphere_100p_00100.fits procesado | Tamaño validado: (70, 70) | SSIM: 0.7721 | PSNR: 24.13 dB
   * Coincidencia hallada: offset(Y=16, X=16). Alineación completada.
✅ ID chromosphere_100p_100_199/chromosphere_100p_00101.fits procesado | Tamaño validado: (70, 70) | SSIM: 0.6402 | PSNR: 26.76 dB
   * Coincidencia hallada: offset(Y=16, X=16). Alineación completada.
✅ ID chromosphere_100p_100_199/chromosphere_100p_00102.fits procesado | Tamaño validado: (70, 70) | SSIM: 0.7199 | PSNR: 22.40 dB
   * Coincidencia hallada: offset(Y=16, X=16). Alineación completada.
✅ ID chromosphere_100p_100_199/chromosphere_100p_00103.fits procesado | Tamaño validado: (70, 70) | SSIM: 0.7965 | PSNR: 30.97 dB
   * Coincidencia hallada: offset(Y=16, X=16). Alineación completada.
✅ ID chromosphere_100p_100_199/chromosphere_100p_00104.fits procesado | Tamaño validado: (70, 70) | SSIM: 0.6942 | PSNR: 22.06 dB


In [15]:
comparar_directorios(Extra,ch200,"Dataset Extra.csv",False)

   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID chromosphere_200p_0_99/chromosphere_200p_00000.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.8302 | PSNR: 31.64 dB
   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID chromosphere_200p_0_99/chromosphere_200p_00001.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.7831 | PSNR: 27.36 dB
   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID chromosphere_200p_0_99/chromosphere_200p_00002.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.3769 | PSNR: 5.45 dB
   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID chromosphere_200p_0_99/chromosphere_200p_00003.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.7691 | PSNR: 26.19 dB
   * Coincidencia hallada: offset(Y=11, X=11). Alineación completada.
✅ ID chromosphere_200p_0_99/chromosphere_200p_00004.fits procesado | Tamaño validado: (177, 177) | SSIM: 0.7512 | PSNR: 28.26 dB
   * C